# Evaluate quality using LLM and code-based scorers

## Scale your human expertise with LLM scorers for automated quality evaluation that is tuned to your use case

LLM scorers are AI-powered quality assessment tools that scale human expertise to evaluate GenAI quality automatically - in development and production. They assess semantic correctness, style, safety, and relative quality - answering questions like "Does this answer correctly?" and "Is this appropriate for our brand?"

MLflow has 2 flavors of judges:

- **Built-in Judges** - Research-backed judges for safety, hallucination, retrieval quality, and relevance
- **Custom Judges** - Tune our research-backed LLM judges to your business needs and human expert judgment

MLflow also supports _custom code-based metrics_, so if the built-in judges don't fit your use case, you can write your own.

The same judges can be used to both evaluate quality in development and monitor quality in production.

![demo-eval](https://i.imgur.com/M3kLBHF.gif)

This notebook demonstrates how to create and use LLM judges to evaluate the BGP Banking Customer Support Agent's quality.

## Install packages (only required if running in a Databricks Notebook)

In [ ]:
%pip install -U -r ../requirements.txt
dbutils.library.restartPython()

## Environment Setup

Load environment variables and verify MLflow configuration.

In [ ]:
import sys
sys.path.append('../')
sys.path.append('../../')

import os
import mlflow
from mlflow_demo.utils import *

# Configure for the current participant: derives the object prefix, prompt name,
# model service, and experiment from YOUR username. The widgets only carry the
# shared catalog/schema/warehouse.
setup_databricks_notebook_env()

print('=== Environment Setup (derived from your username) ===')
print(f"prefix:        {os.environ['UC_TOOL_PREFIX']}")
print(f"catalog:       {os.environ['UC_CATALOG']}.{os.environ['UC_SCHEMA']}")
print(f"prompt:        {os.environ['PROMPT_NAME']}")
print(f"model service: {os.environ['LLM_MODEL_SERVICE']}")
print(f"experiment:    {os.environ['MLFLOW_EXPERIMENT_NAME']} (ID {os.environ['MLFLOW_EXPERIMENT_ID']})")
print(f"warehouse:     {os.environ['MLFLOW_TRACING_SQL_WAREHOUSE_ID']}")

import logging
logging.getLogger("urllib3").setLevel(logging.ERROR)
logging.getLogger("mlflow").setLevel(logging.ERROR)

In [ ]:
# Import demo utilities and banking agent evaluators
from mlflow_demo.utils.mlflow_helpers import (
    get_mlflow_experiment_id, generate_evaluation_links
)
from mlflow_demo.evaluation.evaluator import (
    SCORERS, tone, safety, relevance_to_query,
    accuracy, relevance, actionability, response_is_grounded
)

print('All imports successful')

# Step 1: Understanding MLflow Evaluation

A **scorer** is what looks at a trace and performs evaluation on that trace, then returns feedback which gets attached to the trace. This is the core building block of MLflow's evaluation system.

### What's Inside a Scorer?

Within a scorer, you can use:

- **LM judges** - Feed aspects of the trace to an LLM to perform assessments (e.g., "Is this response safe?")
- **Deterministic code** - Count tokens, measure latency, check formatting, validate compliance
- **Combination of both** - Use LLM reasoning for content quality + code for objective measures

### The Complete Workflow: Scorer -> Evaluate -> Monitor

**1. Create Scorers** - Build evaluation logic for your quality dimensions

**2. Run offline evaluation with `mlflow.genai.evaluate()`** - MLflow handles the coordination

**3. Use for Online Monitoring** - Use the same scorers to monitor production quality (notebook 5)

**Run the next cells to see different types of scorers in action**

# Step 2: Predefined LLM Judge Scorers

MLflow provides research-backed judges for common evaluation needs. Here, we will use built-in judge scorers that apply to our banking customer support use case.

`RelevanceToQuery`: Does the agent's response directly address the customer's banking question?

- Checks if the response stays focused on what the customer asked
- Critical for maintaining trust in financial communications

`Safety`: Does the response avoid harmful or inappropriate content?

- Catches potential toxic content or unauthorized financial advice
- Critical safeguard for any customer-facing banking communications

`tone` (Guidelines): Does the response maintain a professional banking tone?

- Ensures the response is appropriate for a BGP Bank customer support assistant
- Verifies knowledgeable, professional tone suitable for financial services

**Documentation**

- [**Predefined Judge Scorers**](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/predefined-judge-scorers)

In [ ]:
# Load recent traces tagged as sample data and evaluate with predefined scorers
from mlflow.genai.scorers import Safety, RelevanceToQuery

print('Loading recent traces from the banking agent...')

traces = mlflow.search_traces(
    max_results=3,
    filter_string='tags.sample_data = "yes"',
    order_by=['timestamp DESC'],
)
print(f'Found {len(traces)} traces for evaluation')

# Run evaluation with predefined scorers + banking tone guideline
eval_results = mlflow.genai.evaluate(
    data=traces,
    scorers=[Safety(), RelevanceToQuery(), tone]
)

print(f'\nEvaluation completed!')
print(f'Run ID: {eval_results.run_id}')

# Generate and display evaluation links
generate_evaluation_links(eval_results.run_id)

# Step 3: Custom Judges using `make_judge()`

While built-in judges handle common quality aspects, your business has specific requirements. We'll use custom judges created with `make_judge()` for banking-specific evaluation.

These judges are defined in `mlflow_demo/evaluation/evaluator.py` and are tuned for the BGP Banking use case:

1. **`accuracy`** - Verifies all account numbers, balances, and transaction amounts match tool output
2. **`relevance`** - Checks the response directly addresses the specific banking question asked
3. **`actionability`** - Evaluates whether the response provides actionable guidance (e.g., how to dispute a transaction)
4. **`response_is_grounded`** - Ensures the response is grounded in tool call results, not hallucinated

**Documentation**

- [**Custom Scorers Documentation**](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers)

In [ ]:
# Run evaluation with banking custom judges from evaluator.py
print('Loading recent traces from the banking agent...')

traces = mlflow.search_traces(
    max_results=3,
    filter_string='tags.sample_data = "yes"',
    order_by=['timestamp DESC'],
)
print(f'Found {len(traces)} traces for evaluation')

# Use the custom judges defined in evaluator.py
custom_judges = [accuracy, relevance, actionability, response_is_grounded]

eval_results = mlflow.genai.evaluate(
    data=traces,
    scorers=custom_judges
)

print(f'\nEvaluation completed!')
print(f'Run ID: {eval_results.run_id}')

# Generate and display evaluation links
generate_evaluation_links(eval_results.run_id)

# Step 4: Running All Scorers Together

You can combine predefined scorers and custom judges in a single evaluation run. The `SCORERS` list in `evaluator.py` includes all seven scorers.

**Run the next cell to evaluate with all scorers at once.**

In [ ]:
# Run evaluation with ALL banking scorers — over ALL sample traces.
# This run attaches the judges' assessments to every sample trace; the later
# notebooks (find & fix, human review, optimization) search on those assessments,
# so run this cell over the full set, not a sample of 3.
print('Loading ALL sample traces from the banking agent...')

traces = mlflow.search_traces(
    max_results=100,
    filter_string='tags.sample_data = "yes"',
    order_by=['timestamp DESC'],
)
print(f'Found {len(traces)} traces for evaluation')

# SCORERS includes: tone, safety, relevance_to_query, accuracy,
#   relevance, actionability, response_is_grounded
eval_results = mlflow.genai.evaluate(
    data=traces,
    scorers=SCORERS
)

print(f'\nEvaluation completed!')
print(f'Run ID: {eval_results.run_id}')

# Generate and display evaluation links
generate_evaluation_links(eval_results.run_id)

#### Keep production monitoring from re-judging these traces

The assessments you just attached are part of the workshop baseline. The tag below
records "traces evaluated up to now" so the monitoring scorer (Notebook 5) only
judges *future* traces instead of re-paying for these again.

In [ ]:
# Mark the experiment as evaluated-up-to-now so monitoring only scores new traces.
from mlflow import MlflowClient

client = MlflowClient()
all_traces = mlflow.search_traces(max_results=1000, return_type='list')
latest_ms = max(t.info.timestamp_ms for t in all_traces) if all_traces else 0
client.set_experiment_tag(
    experiment_id=os.environ['MLFLOW_EXPERIMENT_ID'],
    key='mlflow.latestTraceEvaluationTimestampMs',
    value=str(latest_ms),
)
print(f"Set latestTraceEvaluationTimestampMs={latest_ms} — sample traces are baselined.")

# Step 5: Custom Prompt-based Judge

For complex, nuanced evaluations that require sophisticated reasoning, you can create custom prompt-based judges with full control over:

1. **Judge prompts** - Custom evaluation instructions
2. **Output formats** - Multiple value types (scores, categories, detailed feedback)
3. **Complex logic** - Multi-dimensional assessment

Here, we will create a `customer_satisfaction` judge that evaluates whether the response would satisfy a banking customer.

**Documentation**

- [**Prompt-based Scorers Documentation**](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge/create-prompt-judge)

In [ ]:
# Custom Prompt-based Judge: customer_satisfaction
from mlflow.genai.scorers import scorer
from mlflow.genai.judges import custom_prompt_judge
import json

@scorer
def customer_satisfaction(trace):
    """Assess whether the response would satisfy a banking customer."""
    satisfaction_prompt = """
You are a customer satisfaction analyst evaluating a BGP Bank support response.

AGENT RESPONSE: {{response}}
USER QUESTION: {{question}}

Evaluate whether this response would satisfy a banking customer:
1. Does it directly answer the customer's question?
2. Is the tone professional and empathetic?
3. Does it reference specific account data when relevant?
4. Does it offer next steps or guidance?
5. Would the customer need to follow up or call back?

Choose one:
[[high]]: Customer would be highly satisfied - complete, accurate, helpful
[[medium]]: Customer would be somewhat satisfied - answers the question but could be better
[[low]]: Customer would be dissatisfied - incomplete or unhelpful
[[negative]]: Customer would be frustrated - wrong or misleading information
"""

    # Extract question and response from the trace
    inputs = trace.data.request or ''
    outputs = trace.data.response or ''

    judge = custom_prompt_judge(
        name='customer_satisfaction',
        prompt_template=satisfaction_prompt,
        numeric_values={'high': 4, 'medium': 3, 'low': 2, 'negative': 1}
    )
    return judge(response=outputs, question=inputs)

print('Loading traces for customer satisfaction evaluation...')
traces = mlflow.search_traces(
    max_results=3,
    filter_string='tags.sample_data = "yes"',
    order_by=['timestamp DESC'],
)
print(f'Found {len(traces)} traces for evaluation')

eval_results = mlflow.genai.evaluate(
    data=traces,
    scorers=[customer_satisfaction]
)

print(f'\nEvaluation completed!')
print(f'Run ID: {eval_results.run_id}')
generate_evaluation_links(eval_results.run_id)

# Step 6: Code-based Scorers

For deterministic, measurable criteria that don't require LLM reasoning, code-based scorers provide:

1. **Fast execution** - No LLM API calls
2. **Deterministic results** - Same input always produces same output
3. **Cost-effective** - No token usage costs
4. **Precise control** - Exact logic for measurable criteria

We'll create a `banking_response_compliance` scorer that checks:
- The response mentions specific amounts or dates
- The response references at least one tool result
- The response includes a recommendation or next step

**Documentation**

- [**Code-based Scorers Documentation**](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers)

In [ ]:
# Code-based Scorer: banking_response_compliance
from mlflow.genai.scorers import scorer
from mlflow.entities.assessment import Feedback
import re

@scorer
def banking_response_compliance(trace):
    """Check banking response compliance: specific data, tool references, next steps."""
    response = trace.data.response or ''
    if isinstance(response, str):
        try:
            response_data = json.loads(response)
            response_text = response_data.get('response', response)
        except json.JSONDecodeError:
            response_text = response
    else:
        response_text = str(response)

    score = 0
    max_score = 3
    issues = []

    # 1. Check for specific amounts or dates (e.g., $1,234.56, 2024-01-15)
    amount_pattern = r'\$[\d,]+\.?\d*|\b\d{4}-\d{2}-\d{2}\b'
    if re.search(amount_pattern, response_text):
        score += 1
    else:
        issues.append('No specific amounts or dates found')

    # 2. Check that at least one tool result is referenced
    tool_spans = trace.search_spans(span_type='TOOL')
    if tool_spans and len(tool_spans) > 0:
        score += 1
    else:
        issues.append('No tool results referenced')

    # 3. Check for a recommendation or next step
    next_step_patterns = [
        r'\brecommend\b', r'\bsuggest\b', r'\bnext step\b',
        r'\byou can\b', r'\byou should\b', r'\bplease\b',
        r'\bcontact\b', r'\bapply\b', r'\bdispute\b'
    ]
    if any(re.search(p, response_text, re.IGNORECASE) for p in next_step_patterns):
        score += 1
    else:
        issues.append('No recommendation or next step found')

    normalized = score / max_score
    return Feedback(
        value=normalized,
        rationale=f'Banking compliance: {score}/{max_score}. Issues: {"; ".join(issues) if issues else "None"}'
    )

print('Loading traces for compliance evaluation...')
traces = mlflow.search_traces(
    max_results=3,
    filter_string='tags.sample_data = "yes"',
    order_by=['timestamp DESC'],
)
print(f'Found {len(traces)} traces for evaluation')

eval_results = mlflow.genai.evaluate(
    data=traces,
    scorers=[banking_response_compliance]
)

print(f'\nEvaluation completed!')
print(f'Run ID: {eval_results.run_id}')
generate_evaluation_links(eval_results.run_id)

# Complete MLflow Evaluation Mastery - Summary & Next Steps

Congratulations! You've successfully mastered MLflow's complete evaluation ecosystem for the BGP Banking Customer Support Agent.

## What You've Accomplished

- **Understood the Architecture** - Judges vs Scorers and how they work together
- **Used Predefined Scorers** - Applied `Safety`, `RelevanceToQuery`, and `tone` on banking traces
- **Created Custom Judges** - Used `make_judge()` for banking-specific evaluation (accuracy, relevance, actionability, groundedness)
- **Implemented Prompt-based Judges** - Created a `customer_satisfaction` judge for banking
- **Developed Code-based Scorers** - Built `banking_response_compliance` for deterministic checks

## Resources for Continued Learning

- [**Complete Judge & Scorer Reference**](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/)
- [**Evaluation Best Practices**](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/evaluate-app)
- [**Monitoring & Alerting**](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/monitor-app)

## Exercise

Write a new Guidelines scorer: _"The response must not expose full account numbers -- only last 4 digits."_ Run evaluation.

Hint: Use `Guidelines(name='account_privacy', guidelines='...')` and add it to the scorers list when calling `mlflow.genai.evaluate()`.

## Exercise

Modify the accuracy judge instructions to be stricter about transaction amounts. Re-run evaluation. How does the pass rate change?

Hint: Import `make_judge` from `mlflow.genai.judges` and create a new judge with stricter instructions, or modify the instructions in `mlflow_demo/evaluation/evaluator.py`.

## Exercise

Write a code-based scorer that checks if the response mentions at least one specific dollar amount.

Hint: Use a regex pattern like `r'\$[\d,]+\.?\d*'` to detect dollar amounts in the response text. Return a `Feedback` object with `value=True` or `value=False`.